In [3]:
# main.ipynb

# Install necessary packages if not already installed
# !pip install transformers chromadb sentence-transformers Levenshtein

# Import modules
import sys
sys.path.append('./')  # Adjust the path if necessary

from data_loader import load_user_reviews
from model_pipeline import RecommenderModel
from utils import extract_latest_n_reviews, extract_product_names
from retrieval import initialize_chromadb, get_or_create_collection, collect_results_alternating_shortest
from evaluation import normalize, compute_similarity, recall_at_k, ndcg_at_k

# Additional imports
import numpy as np
import torch
import pandas as pd

# Initialize Recommender Model
recommender_model = RecommenderModel()

# Initialize ChromaDB
db_path = "./chroma_db_mpnet"
db = initialize_chromadb(db_path)
collection_name = 'product_embeddings'  # Use the same collection name
collection = get_or_create_collection(db, collection_name)

# Load the product data
df = pd.read_csv("data/meta_all_beauty_filtered_simple.csv")

# Load training data
train_file = 'data/train_val_user_reviews.json'
input_set = load_user_reviews(train_file)

# Load test data
test_file = 'data/test_user_reviews.json'
input_set_test = load_user_reviews(test_file)

# Parameters
n_latest_reviews = 3
K = 10  # For Recall@K and NDCG@K
SIMILARITY_THRESHOLD = 90.0

# Initialize lists to store overall results
all_similarity_scores = []
all_matches = []
all_recalls = []
all_ndcgs = []

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
Unused kwargs: ['_load_in_4bit', '_load_in_8bit', 'quant_method']. These kwargs are not used in <class 'transformers.utils.quantization_config.BitsAndBytesConfig'>.


models/hf-frompretrained-download/meta-llama/Meta-Llama-3-8B-Instruct


`low_cpu_mem_usage` was None, now set to True since model is quantized.
Loading checkpoint shards: 100%|██████████| 2/2 [00:02<00:00,  1.24s/it]


In [1]:
# Open a file to save the results
with open('results.txt', 'w', encoding='utf-8') as result_file:
    skipped_users = []  # List to keep track of skipped user IDs

    # Loop over all users in the test set
    num_users = len(input_set_test)
    for userIndex in range(num_users):
        print(f"Processing user {userIndex + 1}/{num_users}")

        # Extract latest reviews for training
        example_user = [input_set[userIndex]]
        latest_reviews = extract_latest_n_reviews(example_user, n_latest_reviews)
        review_text = "\n".join([rev['text'] for rev in latest_reviews])

        # Create user profile
        profile = recommender_model.create_user_profile(review_text)

        # Generate preliminary recommendations
        preliminary_rec = recommender_model.create_preliminary_recommendations(profile)

        # Extract product names
        product_names = extract_product_names(preliminary_rec)

        # Check if product_names is empty
        if not product_names:
            print(f"User {userIndex + 1} has empty product names. Skipping.")
            result_file.write(f"User {userIndex + 1} skipped due to empty product names.\n\n")
            skipped_users.append(userIndex + 1)
            continue  # Skip to the next user

        # Query ChromaDB and collect results
        final_results = collect_results_alternating_shortest(product_names, collection)

        # Check if collect_results_alternating_shortest returned -1
        if final_results == -1:
            print(f"User {userIndex + 1} has no recommendations. Skipping.")
            result_file.write(f"User {userIndex + 1} skipped due to no recommendations.\n\n")
            skipped_users.append(userIndex + 1)
            continue  # Skip to the next user

        # Proceed with evaluation if we have recommendations
        # Get the actual product from the test set
        example_user_test = [input_set_test[userIndex]]
        test_review = extract_latest_n_reviews(example_user_test, 1)
        test_product = test_review[0]['product_name']

        # Evaluate recommendations
        normalized_test_product = normalize(test_product)
        recommended_products = []
        for doc, metadata in final_results:
            # Retrieve the product title using metadata (e.g., 'asin')
            asin = metadata.get('asin')
            filt = df['asin'] == asin
            title = df.loc[filt, 'title'].values
            if len(title) > 0:
                recommended_products.append(title[0])
            else:
                recommended_products.append(doc)  # Fallback to the document if title not found

        normalized_recommended_products = [normalize(name) for name in recommended_products]

        # Compute similarity scores and matches
        similarity_scores = []
        matches = []
        for rec_product in normalized_recommended_products:
            sim_score = compute_similarity(rec_product, normalized_test_product)
            similarity_scores.append(sim_score)
            matches.append(sim_score >= SIMILARITY_THRESHOLD)

        # Save results for this user
        result_file.write(f"User {userIndex + 1}:\n")
        result_file.write(f"Test Product: {test_product}\n")
        result_file.write(f"Recommended Products:\n")
        for i, (prod, score, match) in enumerate(zip(recommended_products, similarity_scores, matches)):
            result_file.write(f"  {i+1}. {prod} - Similarity: {score:.2f}% - {'Match' if match else 'No Match'}\n")
        result_file.write("\n")

        # Collect similarity scores and matches
        all_similarity_scores.extend(similarity_scores)
        all_matches.extend(matches)

        # Calculate Recall@K and NDCG@K for this user
        recall = recall_at_k(matches, K)
        ndcg = ndcg_at_k(matches, K)
        all_recalls.append(recall)
        all_ndcgs.append(ndcg)

        # Optional: Print progress
        print(f"User {userIndex + 1} - Recall@{K}: {recall}, NDCG@{K}: {ndcg}")

    # Calculate overall metrics
    if all_recalls and all_ndcgs:
        mean_recall = np.mean(all_recalls)
        mean_ndcg = np.mean(all_ndcgs)
    else:
        mean_recall = 0.0
        mean_ndcg = 0.0

    # Write overall metrics and skipped users to the file
    result_file.write("Overall Evaluation Metrics:\n")
    result_file.write(f"Mean Recall@{K}: {mean_recall}\n")
    result_file.write(f"Mean NDCG@{K}: {mean_ndcg}\n\n")

    if skipped_users:
        result_file.write("Skipped Users:\n")
        result_file.write(", ".join(map(str, skipped_users)))
        result_file.write("\n")

# Print overall metrics
print("\nOverall Evaluation Metrics:")
print(f"Mean Recall@{K}: {mean_recall}")
print(f"Mean NDCG@{K}: {mean_ndcg}")

NameError: name 'input_set_test' is not defined

In [11]:
with open('results.txt', 'w', encoding='utf-8') as result_file:
    skipped_users = []  # List to keep track of skipped user IDs

    # Loop over all users in the test set
    num_users = len(input_set_test)
    for userIndex in range(num_users):
        print(f"Processing user {userIndex + 1}/{num_users}")
        
        # Extract latest reviews for training
        example_user = [input_set[userIndex]]
        latest_reviews = extract_latest_n_reviews(example_user, n_latest_reviews)
        #review_text = "\n".join([rev['review_text'] for rev in latest_reviews])

        # Create user profile
        profile = recommender_model.create_user_profile(latest_reviews)
        
        # Generate preliminary recommendations
        preliminary_rec = recommender_model.create_preliminary_recommendations(profile)
        
        # Extract product names
        product_names = extract_product_names(preliminary_rec)

        # Check if product_names is empty
        if not product_names:
            print(f"User {userIndex + 1} has empty product names. Skipping.")
            result_file.write(f"User {userIndex + 1} skipped due to empty product names.\n\n")
            skipped_users.append(userIndex + 1)
            continue  # Skip to the next user

        # Query ChromaDB and collect results
        final_list = collect_results_alternating_shortest(product_names, collection)

        # Check if collect_results_alternating_shortest returned -1
        if final_list == -1:
            print(f"User {userIndex + 1} has no recommendations. Skipping.")
            result_file.write(f"User {userIndex + 1} skipped due to no recommendations.\n\n")
            skipped_users.append(userIndex + 1)
            continue  # Skip to the next user

        # Proceed with evaluation if we have recommendations
        # Get the actual product from the test set
        example_user_test = [input_set_test[userIndex]]
        test_review = extract_latest_n_reviews(example_user_test, 1)
        test_product = test_review[0]['product_name']
        
        # Evaluate recommendations
        normalized_test_product = normalize(test_product)
        normalized_recommended_products = [normalize(name) for name in final_list]
        
        # Compute similarity scores and matches
        similarity_scores = []
        matches = []
        for rec_product in normalized_recommended_products:
            sim_score = compute_similarity(rec_product, normalized_test_product)
            similarity_scores.append(sim_score)
            matches.append(sim_score >= SIMILARITY_THRESHOLD)
        
        # Save results for this user
        result_file.write(f"User {userIndex + 1}:\n")
        result_file.write(f"Test Product: {test_product}\n")
        result_file.write(f"Recommended Products:\n")
        for i, (prod, score, match) in enumerate(zip(final_list, similarity_scores, matches)):
            result_file.write(f"  {i+1}. {prod} - Similarity: {score:.2f}% - {'Match' if match else 'No Match'}\n")
        result_file.write("\n")
        
        # Collect similarity scores and matches
        all_similarity_scores.extend(similarity_scores)
        all_matches.extend(matches)
        
        # Calculate Recall@K and NDCG@K for this user
        recall = recall_at_k(matches, K)
        ndcg = ndcg_at_k(matches, K)
        all_recalls.append(recall)
        all_ndcgs.append(ndcg)
        
        # Optional: Print progress
        print(f"User {userIndex + 1} - Recall@{K}: {recall}, NDCG@{K}: {ndcg}")
    
    # Calculate overall metrics
    if all_recalls and all_ndcgs:
        mean_recall = np.mean(all_recalls)
        mean_ndcg = np.mean(all_ndcgs)
    else:
        mean_recall = 0.0
        mean_ndcg = 0.0
    
    # Write overall metrics and skipped users to the file
    result_file.write("Overall Evaluation Metrics:\n")
    result_file.write(f"Mean Recall@{K}: {mean_recall}\n")
    result_file.write(f"Mean NDCG@{K}: {mean_ndcg}\n\n")
    
    if skipped_users:
        result_file.write("Skipped Users:\n")
        result_file.write(", ".join(map(str, skipped_users)))
        result_file.write("\n")
    
# Print overall metrics
print("\nOverall Evaluation Metrics:")
print(f"Mean Recall@{K}: {mean_recall}")
print(f"Mean NDCG@{K}: {mean_ndcg}")

Processing user 1/253
User 1 - Recall@10: 0.0, NDCG@10: 0.0
Processing user 2/253
User 2 - Recall@10: 0.0, NDCG@10: 0.0
Processing user 3/253
User 3 - Recall@10: 0.0, NDCG@10: 0.0
Processing user 4/253
User 4 - Recall@10: 0.0, NDCG@10: 0.0
Processing user 5/253
User 5 - Recall@10: 0.0, NDCG@10: 0.0
Processing user 6/253
User 6 has empty product names. Skipping.
Processing user 7/253
User 7 - Recall@10: 0.0, NDCG@10: 0.0
Processing user 8/253
User 8 has empty product names. Skipping.
Processing user 9/253
User 9 - Recall@10: 0.0, NDCG@10: 0.0
Processing user 10/253
User 10 - Recall@10: 0.0, NDCG@10: 0.0
Processing user 11/253
User 11 - Recall@10: 0.0, NDCG@10: 0.0
Processing user 12/253
User 12 - Recall@10: 0.0, NDCG@10: 0.0
Processing user 13/253
User 13 - Recall@10: 0.0, NDCG@10: 0.0
Processing user 14/253
User 14 has empty product names. Skipping.
Processing user 15/253
User 15 - Recall@10: 0.0, NDCG@10: 0.0
Processing user 16/253
User 16 - Recall@10: 0.0, NDCG@10: 0.0
Processing use

ValueError: Input length of input_ids is 2048, but `max_length` is set to 2048. This can lead to unexpected behavior. You should consider increasing `max_length` or, better yet, setting `max_new_tokens`.